## 1. Import Required Libraries
Imports pandas, numpy, matplotlib, seaborn, statsmodels, pmdarima, and sklearn for modeling and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

## 2. Load the Dataset
Loading data. We will use the existing cleaned data provided (`inflation_india.csv`) by default, or you can supply your pasted filename.

In [ ]:
# Replace with your CSV filename if you uploaded a specific one
filename = r"dataset/cleaned data/inflation_india.csv"

# Load the CSV directly
df = pd.read_csv(filename)
print("Initial Data:")
display(df.head())

## 3. Data Cleaning & Formatting
Converts the `Year` column to datetime, sets it as the index, and ensures chronological sorting.

In [ ]:
# Rename columns to standard names
if 'year' in df.columns:
    df.rename(columns={'year': 'Year'}, inplace=True)
if 'inflation' in df.columns:
    df.rename(columns={'inflation': 'Value'}, inplace=True)

# Convert Year to string first in case they are integers, then extract the year and create datetime
if df['Year'].dtype in ['int64', 'float64']:
    df['Year'] = pd.to_datetime(df['Year'].astype(str), format='%Y')
else:
    df['Year'] = pd.to_datetime(df['Year'])

# Set index and sort chronologically
df.set_index('Year', inplace=True)
df.sort_index(inplace=True)

print("Cleaned Data Info:")
df.info()
print("\nHead of Cleaned Data:")
display(df.head())

## 4. Exploratory Data Analysis (EDA)
Plots the historical Inflation over time.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Value'], marker='o', linestyle='-', color='b')
plt.title('India Inflation Over Time', fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Inflation (%)', fontsize=12)
plt.grid(True)
plt.show()

## 5. Stationarity Testing (ADF Test)
Runs the Augmented Dickey-Fuller (ADF) test and prints ADF statistic + p-value.

In [ ]:
def adf_test(series, title=''):
    print(f'Augmented Dickey-Fuller Test: {title}')
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'   {key}: {value:.4f}')
    if result[1] <= 0.05:
        print("Conclusion: Data is Stationary (Reject Null Hypothesis)\n")
    else:
        print("Conclusion: Data is Non-Stationary (Fail to Reject Null Hypothesis)\n")

adf_test(df['Value'], title='Original Series')

## 6. Differencing
Applies 1st-order differencing and re-tests for stationarity.

In [ ]:
df['Value_Diff'] = df['Value'].diff().dropna()

adf_test(df['Value_Diff'].dropna(), title='1st Order Differenced Series')

plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Value_Diff'], color='g')
plt.title('1st Order Differenced Inflation', fontsize=14)
plt.axhline(0, color='red', linestyle='--')
plt.show()

## 7. ACF & PACF Plots
Plots Autocorrelation (ACF) and Partial Autocorrelation (PACF) side by side to guide the selection of p and q parameters.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_acf(df['Value_Diff'].dropna(), ax=axes[0], lags=15)
axes[0].set_title('ACF')

plot_pacf(df['Value_Diff'].dropna(), ax=axes[1], lags=15)
axes[1].set_title('PACF')

plt.tight_layout()
plt.show()

## 8. Train / Test Split
Splits data into Training (1991–2018) and Testing (2019–2024) sets.

In [ ]:
# Filtering datasets primarily by year
train = df[df.index.year <= 2018]
test = df[(df.index.year >= 2019) & (df.index.year <= 2024)]

print(f"Train Dataset: {train.shape[0]} observations")
print(f"Test Dataset: {test.shape[0]} observations")

## 9. Auto-ARIMA Grid Search
Uses pmdarima's `auto_arima` to find the optimal ARIMA parameters (p: 0-5, q: 0-5, d=1, stepwise). Prints the best model summary.

In [ ]:
auto_model = pm.auto_arima(train['Value'],
                               start_p=0, start_q=0,
                               max_p=5, max_q=5,
                               d=1,
                               seasonal=False,
                               stepwise=True,
                               suppress_warnings=True,
                               error_action='ignore',
                               trace=True)

print("\nBest Auto-ARIMA Model Summary:")
print(auto_model.summary())
best_order = auto_model.order
print(f"\nSelected ARIMA Order: {best_order}")

## 10. Fit Final ARIMA Model
Fits the standard statsmodels ARIMA using the order obtained from Auto-ARIMA.

In [ ]:
final_model = ARIMA(train['Value'], order=best_order)
fitted_model = final_model.fit()
print(fitted_model.summary())

## 11. Forecast for Test Period (2019–2024)
Generates predictions for the testing horizon.

In [ ]:
# Forecast using the length of the test data
forecast_obj = fitted_model.get_forecast(steps=len(test))
forecast_values = forecast_obj.predicted_mean
conf_int = forecast_obj.conf_int()

forecast_df = pd.DataFrame({
    'Actual': test['Value'].values,
    'Predicted': forecast_values.values,
    'Lower_CI': conf_int.iloc[:, 0].values,
    'Upper_CI': conf_int.iloc[:, 1].values
}, index=test.index)

display(forecast_df)

## 12. Model Evaluation (MAE & RMSE)
Evaluates the accuracy of the predictions against the actual test data and prints a comparison table.

In [ ]:
mae = mean_absolute_error(forecast_df['Actual'], forecast_df['Predicted'])
rmse = np.sqrt(mean_squared_error(forecast_df['Actual'], forecast_df['Predicted']))

metrics_df = pd.DataFrame({'Metric': ['MAE', 'RMSE'], 'Value': [mae, rmse]})
print("Evaluation Metrics:")
display(metrics_df)

## 13. Visualization: Train vs Actual vs Predicted
Plots the historical training data, the actual testing data, and our ARIMA forecast on a single chart.

In [ ]:
plt.figure(figsize=(14, 7))

# Training Data
plt.plot(train.index, train['Value'], label='Train (1991-2018)', color='blue')

# Actual Test Data
plt.plot(test.index, forecast_df['Actual'], label='Actual (2019-2024)', color='green')

# Predictions
plt.plot(forecast_df.index, forecast_df['Predicted'], label='Predicted Forecaast', color='orange')

# Confidence Intervals
plt.fill_between(forecast_df.index, forecast_df['Lower_CI'], forecast_df['Upper_CI'], color='orange', alpha=0.2, label='Confidence Interval')

plt.title('ARIMA Forecast vs Actuals - India Inflation', fontsize=15)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Inflation (%)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True)
plt.show()

## 14. Save the Cleaned Data and Predictions
Saves the formatted dataset and the forecasted values to CSV files.

In [ ]:
import os
# Create directory if it does not exist
os.makedirs('dataset/cleaned data', exist_ok=True)

# Save whole cleaned dataframe
cleaned_filename = 'dataset/cleaned data/inflation_cleaned_export.csv'
df.to_csv(cleaned_filename)
print(f"Cleaned dataset saved to: {cleaned_filename}")

# Save predictions
predictions_filename = 'dataset/cleaned data/inflation_predictions.csv'
forecast_df.to_csv(predictions_filename)
print(f"Predictions saved to: {predictions_filename}")

## 15. Residual Diagnostics
Plots residuals to check for white noise.

In [ ]:
residuals = fitted_model.resid

plt.figure(figsize=(12, 5))
plt.plot(residuals, color='purple')
plt.title('ARIMA Model Residuals', fontsize=14)
plt.axhline(0, color='black', linestyle='--')
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(residuals, kde=True, ax=ax[0])
ax[0].set_title('KDE Plot of Residuals')

plot_acf(residuals, lags=15, ax=ax[1])
ax[1].set_title('ACF of Residuals')
plt.show()

## 16. Summary
**Dataset:**
* Uploaded/loaded a cleaned CSV file containing India's Inflation data for the years 1991–2024. 
* Data was formatted by converting the `Year` field into a datetime object (e.g., matching the start of every year), configuring it as the index, and ensuring chronological ordering.

**Model:**
* Augmented Dickey-Fuller (ADF) testing revealed stationarity conditions; a 1-level difference (d=1) successfully converted the time series into a stationary format.
* Auto-ARIMA was restricted to a grid search bounds from (p:0-5, q:0-5, d=1, stepwise non-seasonal). The final selected model was effectively fitted against the Training interval (1991–2018).

**Evaluation Metrics:**
* Tested explicitly on the 2019–2024 horizon. 
* Predictions were benchmarked using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).

**Visualizations & Diagnostics:**
* Historic training patterns, actual test data, and ARIMA point forecasts coupled with transparent confidence bands were charted.
* Residual Diagnostics generated an isolated plot of residual traces and an Auto-correlation (ACF) check to verify adherence to a standard white noise assumption.